## Basic NeuroDOT Parcel-based Functional Connectivity Script

In [ ]:
pip install neurodot_py

In [ ]:
pip install -r requirements.txt 

In [ ]:

import numpy as np
import neuro_dot as ndot
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.offline as po

# =============================================================================
# Load data and necessary support files
# =============================================================================
# Add the full path to the data, or have it in your current working directory
data = ndot.loadmat7p3('Subj3_Nph2014_REST-image.mat')
cortex_Hb = data['cortex_Hb']
info = data['info']

# Add the full paths to each of the support files, or have them in your current working directory
A = ndot.loadmat7p3('A_Adult_96x92.mat')
Overlap = ndot.loadmat('Adult_96x92_Overlapped_Gordon_Parcels.mat')
Overlap_parcel = Overlap['Overlap_parcel']
FOVfile = ndot.loadmat('Adult_96x92_FOV.mat')
FOV = FOVfile['FOV']
[MNI_parcels, parcelsInfo] = ndot.LoadVolumetricData('Parcels_MNI_333','','nii')
Anat = ndot.loadmat('MNI164k_big.mat');
MNIl = Anat['MNIl']
MNIr = Anat['MNIr']

# =============================================================================
# Prepare HbO data for FC calculation
# =============================================================================

GVTD_th = 1e-3  # Threshold relative to GVTD distribution
t_mask = info["GVTD"] < GVTD_th  # Boolean mask of clean timepoints

Nt2 = t_mask.sum()

HbO = cortex_Hb[:, :, 0]           # shape: (n_voxels, n_timepoints), 0-indexed

HbO_tm = HbO[:, t_mask]            # Kill data outside of t_mask
HbO_m = HbO_tm.mean(axis=1)        # DC / global mean, shape: (n_voxels,)
HbO_v = HbO_tm - HbO_m[:, np.newaxis]  # Subtract mean (broadcast over time)
HbO_v = ndot.normalize_rows(HbO_v)      # Normalize across time (see parcel_based_fc)

HbO_v =ndot.GoodVox2vol(HbO_v, A["info"]["tissue"]["dim"])
HbO_vol = ndot.affine3d_img(
    HbO_v,
    A["info"]["tissue"]["dim"],
    parcelsInfo,
    A["info"]["tissue"]["affine"],
)  # Remap from native dim to MNI space


# =============================================================================
# Calculate Parcel-based Functional Connectivity
# =============================================================================
fc_maps, fc_matrix, parcel_tt = ndot.parcel_based_fc(HbO_vol, Overlap_parcel, parcelsInfo)
fc_maps = fc_maps * FOV[:, :, :, np.newaxis]


# ==========================================================================================================
# Plot example FC maps for parcels of interest NOTE: this will open a figure window in your internet browser
# ==========================================================================================================
# Parcel indices (converted to 0-based)
visual_parcels      = [41, 74]
auditory_parcels    = [23, 95]
somatomotor_parcels = [10, 85]
fp_parcels          = [50, 76]

# Flatten into a list of 8 parcel indices
parcel_list = [
    visual_parcels[0], auditory_parcels[0], somatomotor_parcels[0], fp_parcels[0],  # row 1
    visual_parcels[1], auditory_parcels[1], somatomotor_parcels[1], fp_parcels[1],  # row 2
]
network_labels = ["Visual", "Auditory", "SMhand", "Fronto-Parietal"]

# Suppress po.plot() during loop
original_plot = po.plot
po.plot = lambda *args, **kwargs: None

# Use the first parcel to extract the correct scene layout
parcel_num = parcel_list[0]
fc_map = fc_maps[:, :, :, parcel_num]
plot_params = {"Scale": 0.5 * np.nanmax(fc_map), "Th": {"P": 0, "N": 0}}
_, _, ref_fig, _ = ndot.PlotInterpSurfMesh(
    np.squeeze(fc_map), MNIl, MNIr, parcelsInfo, plot_params
)
ref_scene = ref_fig.layout.scene.to_plotly_json()  # extract as dict

# Build subplot figure
fig = make_subplots(
    rows=2, cols=4,
    specs=[[{'type': 'scene'}] * 4] * 2,
    subplot_titles=[f"Parcel {p + 1}" for p in parcel_list]
)

for i, parcel_num in enumerate(parcel_list):
    row = i // 4 + 1
    col = i % 4 + 1

    fc_map = fc_maps[:, :, :, parcel_num]
    plot_params = {"Scale": 0.5 * np.nanmax(fc_map), "Th": {"P": 0, "N": 0}}

    _, _, sub_fig, _ = ndot.PlotInterpSurfMesh(
        np.squeeze(fc_map), MNIl, MNIr, parcelsInfo, plot_params
    )

    for trace in sub_fig.data:
        fig.add_trace(trace, row=row, col=col)

    # Copy scene layout from reference figure into each subplot
    scene_key = "scene" if i == 0 else f"scene{i + 1}"
    fig.update_layout({scene_key: ref_scene})

# Global layout
network_labels = ["Visual", "Auditory", "SMhand", "Fronto-Parietal"]
x_centers = []
for i in range(4):
    scene_key = "scene" if i == 0 else f"scene{i+1}"
    domain_x = fig.layout[scene_key].domain.x
    x_centers.append((domain_x[0] + domain_x[1]) / 2)

for j, label in enumerate(network_labels):
    fig.add_annotation(
        x=x_centers[j], y=1.13,
        xref="paper", yref="paper",
        text=f"<b>{label}</b>",
        showarrow=False,
        font=dict(color="white", size=14),
        xanchor="center",
    )
fig.update_layout(
    paper_bgcolor="black",
    plot_bgcolor="black",
    font=dict(color="white"),
    height=600,
    width=1400,
    margin=dict(t=100),  # increase top margin
)

# Restore and show only the subplot figure
po.plot = original_plot
fig.show(renderer="browser")



In [ ]:
from scipy.io import savemat

# Save to mat file
outputFileName = 'Parcelbased_FC_output.mat'
savemat(outputFileName, {
    'fc_maps':    fc_maps,
    'fc_matrix':  fc_matrix,
    'parcel_tt':  parcel_tt
})